# Variant 9 — Full Reasoning LoRA + 5-Candidate Majority Verifier

Pipeline dựa trên đề xuất trong `gpt2.md`: giữ lời giải gốc dài hơn để model học chuỗi suy luận, tăng độ dài sinh, sinh 5 candidate, dùng majority voting thật sự và verifier fallback.

Mục tiêu: cải thiện Variant 6 bằng cách tránh target quá ngắn/mất suy luận, đồng thời giảm lỗi sinh cụt trước anchor `Đáp án là:`.

# Cell 1 — Setup, config, paths

Cấu hình Kaggle path, seed, giới hạn dữ liệu, LoRA và training cho `variant_9_full_reasoning_multi_cand`.

In [1]:
# Cell 1 — Setup, config, paths

import os
import re
import gc
import json
import math
import random
import hashlib
import inspect
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ======================
# Kaggle input paths
# ======================
TRAIN_PATH = Path("/kaggle/input/datasets/kimanh2002/dataset-math/train.json")
VALID_PATH = Path("/kaggle/input/datasets/kimanh2002/dataset-math/valid.json")
TEST_PATH  = Path("/kaggle/input/datasets/kimanh2002/dataset-math/test.json")

# Base model path. If this exact path is different on Kaggle, the function find_model_path() below will try to locate it.
MODEL_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlp-hust-gpt2-vietnamese"),
    Path("/kaggle/input/gpt2-vietnamese"),
]

WORK_DIR = Path("/kaggle/working")
PROC_DIR = WORK_DIR / "processed_variant9_full_reasoning"
MODEL_OUT_DIR = WORK_DIR / "lora_gpt2_math_variant9"
PRED_DIR = WORK_DIR / "predictions"

for p in [PROC_DIR, MODEL_OUT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SAFE_EOS_ID = 50256

# ======================
# Main knobs for 3-hour Kaggle run
# ======================
VARIANT_NAME = "variant_9_full_reasoning_multi_cand"
TRAIN_MAX_SAMPLES = 50000       # Theo gpt2.md: giữ 50k mẫu sạch để vẫn trong giới hạn 3h.
VALID_MAX_SAMPLES = None        # None = evaluate full valid. Use 1000 for quick debug.
MAX_QUERY_CHARS = 900
MAX_RESPONSE_CHARS = 1400
MAX_TOTAL_CHARS = 2500

MAX_LENGTH = 512                # prompt + full-reasoning target token length
GEN_MAX_NEW_TOKENS = 128

# LoRA/training config
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
EVAL_STEPS = 250
SAVE_STEPS = 500
LOGGING_STEPS = 25

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("TRAIN_PATH exists:", TRAIN_PATH.exists(), TRAIN_PATH)
print("VALID_PATH exists:", VALID_PATH.exists(), VALID_PATH)

CUDA available: True
GPU: Tesla T4
TRAIN_PATH exists: True /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_PATH exists: True /kaggle/input/datasets/kimanh2002/dataset-math/valid.json


# Cell 2 — I/O helpers and load raw data

In [2]:
# Cell 2 — I/O helpers and load raw data

def read_json_or_jsonl(path: Path) -> Any:
    """Read JSON array/object, JSONL, or concatenated JSON objects."""
    with path.open("r", encoding="utf-8-sig") as f:
        text = f.read().strip()

    if not text:
        return []

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    decoder = json.JSONDecoder()
    records = []
    idx = 0
    n = len(text)

    while idx < n:
        while idx < n and text[idx].isspace():
            idx += 1
        if idx >= n:
            break
        obj, end = decoder.raw_decode(text, idx)
        records.append(obj)
        idx = end

    return records

def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return out

def ensure_list_records(obj: Any) -> List[Dict[str, Any]]:
    if isinstance(obj, list):
        return [x for x in obj if isinstance(x, dict)]
    if isinstance(obj, dict):
        # Some datasets wrap records under a key.
        for key in ["data", "records", "train", "valid", "examples"]:
            if key in obj and isinstance(obj[key], list):
                return [x for x in obj[key] if isinstance(x, dict)]
        return [obj]
    return []

def stable_hash(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    return hashlib.md5(text.encode("utf-8")).hexdigest()

raw_train = ensure_list_records(read_json_or_jsonl(TRAIN_PATH))
raw_valid = ensure_list_records(read_json_or_jsonl(VALID_PATH))

print("raw_train:", len(raw_train))
print("raw_valid:", len(raw_valid))
print("sample keys:", raw_train[0].keys() if raw_train else None)
print(json.dumps(raw_train[0], ensure_ascii=False, indent=2)[:1200])

raw_train: 95400
raw_valid: 1000
sample keys: dict_keys(['original_question_vi', 'original_question_en', 'query_vi', 'query_en', 'response_vi', 'response_en', 'type'])
{
  "original_question_vi": "Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực phẩm luôn chuẩn bị thêm mười đĩa đề phòng trường hợp có sự cố xảy ra. Mỗi đĩa bít tết và măng tây sốt bơ tỏi sẽ có 8 ngọn măng tây trên đó. Người cung cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?",
  "original_question_en": "Bridgette and Alex are getting married. Bridgette is inviting 84 guests, and Alex is inviting two thirds of that number of guests. They hired a caterer to make a plated meal for each guest at the wedding reception. The caterer always makes ten extra plates just in case something goes wrong. Each plate of steak and asparagus in garlic butter will have 8 asparagus spea

# Cell 3 — Evaluator / extraction / parsing
Dùng để verify evaluator trước khi train.

In [3]:
# Cell 3 — Evaluator/extraction/parsing functions

ANSWER_PATTERNS = [
    r"Đáp án là\s*[:：]?\s*([^\n]+)",
    r"Câu trả lời là\s*[:：]?\s*([^\n]+)",
    r"The answer is\s*[:：]?\s*([^\n]+)",
    r"####\s*([^\n]+)",
]

BAD_ANSWER_TOKENS = [
    "nan", "inf", "undefined", "không xác định", "không đủ",
    "unknown", "none", "n/a"
]

def normalize_unicode_text(text: str) -> str:
    text = str(text or "")
    text = text.replace("\ufeff", "").replace("\u200b", "")
    text = text.replace("\u00a0", " ")
    text = text.replace("−", "-")
    text = text.replace("×", "*")
    text = text.replace("÷", "/")
    text = text.replace("\\đóng hộp", "\\boxed")
    text = text.replace("\\ đóng hộp", "\\boxed")
    return text

def strip_boxed(text: str) -> str:
    s = str(text or "")
    # Repeatedly unwrap simple \boxed{...}
    for _ in range(3):
        s2 = re.sub(r"\\boxed\s*\{([^{}]+)\}", r"\1", s)
        if s2 == s:
            break
        s = s2
    return s

def clean_answer_candidate(ans: str) -> str:
    ans = normalize_unicode_text(ans)
    ans = strip_boxed(ans)
    ans = ans.strip()

    # Nếu candidate bị dính thêm anchor khác phía sau, cắt bỏ phần sau.
    ans = re.split(
        r"\s+(?:Đáp án là|Câu trả lời là|The answer is)\s*[:：]?",
        ans,
        flags=re.IGNORECASE
    )[0]

    # Không split bằng dấu "." vì sẽ làm hỏng số thập phân như 4.5.
    ans = re.split(r"(?:。|\n)", ans)[0].strip()

    ans = ans.replace("$", "").replace("`", "").strip()
    ans = re.sub(r"^=+\s*", "", ans)
    ans = re.sub(r"^[\:：]\s*", "", ans)
    ans = re.sub(r"\s+", " ", ans).strip()
    ans = ans.rstrip(".,;:，。)】] ")
    ans = ans.replace("\\ ", "")
    ans = ans.replace("{ ", "{").replace(" }", "}")

    return ans.strip()


def extract_final_answer(text: str, fallback_last_number: bool = True) -> Optional[str]:
    text = normalize_unicode_text(text)

    # Ưu tiên anchor chính trước, không trộn với ####.
    priority_patterns = [
        r"Đáp án là\s*[:：]?\s*([^\n]+)",
        r"Câu trả lời là\s*[:：]?\s*([^\n]+)",
        r"The answer is\s*[:：]?\s*([^\n]+)",
    ]

    for pat in priority_patterns:
        matches = list(re.finditer(pat, text, flags=re.IGNORECASE))
        for m in reversed(matches):
            cand = clean_answer_candidate(m.group(1))
            if cand and not is_bad_final_answer_candidate(cand):
                return cand

    # Sau đó mới fallback sang ####.
    matches = list(re.finditer(r"####\s*([^\n]+)", text, flags=re.IGNORECASE))
    for m in reversed(matches):
        cand = clean_answer_candidate(m.group(1))
        if cand and not is_bad_final_answer_candidate(cand):
            return cand

    if not fallback_last_number:
        return None

    fallback_patterns = [
        r"\\frac\s*\{[-+]?\d+(?:\.\d+)?\}\s*\{[-+]?\d+(?:\.\d+)?\}",
        r"[-+]?\d+(?:[.,]\d+)?\s*\\?pi",
        r"[-+]?\d+(?:[.,]\d+)?\s*/\s*[-+]?\d+(?:[.,]\d+)?",
        r"[-+]?\d+(?:[.,]\d+)?",
    ]

    found = []

    for pat in fallback_patterns:
        found.extend(re.findall(pat, text, flags=re.IGNORECASE))

    if found:
        return clean_answer_candidate(found[-1])

    return None

def normalize_number_punctuation(s: str) -> str:
    s = str(s or "").strip()

    # Convert decimal comma: 4,5 -> 4.5
    if re.fullmatch(r"[-+]?\d+,\d+", s):
        return s.replace(",", ".")

    # Remove thousand separators: 1,000 or 40.320 in Vietnamese contexts.
    if re.fullmatch(r"[-+]?\d{1,3}(,\d{3})+", s):
        return s.replace(",", "")
    if re.fullmatch(r"[-+]?\d{1,3}(\.\d{3})+", s):
        return s.replace(".", "")

    return s

def canonical_answer(ans: Optional[str]) -> str:
    if ans is None:
        return ""
    s = clean_answer_candidate(ans)
    s = normalize_number_punctuation(s)
    s = s.lower()
    s = s.replace(" ", "")
    s = s.replace("\\dfrac", "\\frac")
    s = strip_boxed(s)
    s = s.rstrip(".")
    return s

def is_bad_final_answer_candidate(ans: Optional[str]) -> bool:
    if ans is None:
        return True
    s = canonical_answer(ans)
    if not s:
        return True
    if len(s) > 80:
        return True
    if any(tok in s.lower() for tok in BAD_ANSWER_TOKENS):
        return True

    # Too much natural language means extraction probably captured explanation.
    letters = re.findall(r"[a-zA-ZÀ-ỹ]", s)
    allowed_words = {"pi"}
    s_without_allowed = s
    for w in allowed_words:
        s_without_allowed = s_without_allowed.replace(w, "")
    if len(re.findall(r"[a-zA-ZÀ-ỹ]", s_without_allowed)) > 3:
        return True

    return False

def latex_frac_to_expr(s: str) -> str:
    s = re.sub(
        r"\\frac\s*\{\s*([-+]?\d+(?:\.\d+)?)\s*\}\s*\{\s*([-+]?\d+(?:\.\d+)?)\s*\}",
        r"(\1/\2)",
        s,
    )
    return s

def parse_answer_value(ans: Optional[str]) -> Optional[Any]:
    """Parse scalar or tuple/list answers for local validation."""
    if ans is None:
        return None

    s = canonical_answer(ans)
    if not s:
        return None

    # Tuple/list answer: (91,60)
    if re.fullmatch(r"\(?[-+]?\d+(?:\.\d+)?(?:,[-+]?\d+(?:\.\d+)?)+\)?", s):
        inner = s.strip("()[]")
        parts = inner.split(",")
        try:
            return tuple(float(normalize_number_punctuation(x)) for x in parts)
        except Exception:
            pass

    s = latex_frac_to_expr(s)
    s = s.replace("\\pi", "pi")
    s = s.replace("π", "pi")

    # 36pi -> 36*pi
    s = re.sub(r"(\d)(pi)", r"\1*\2", s)
    s = re.sub(r"(pi)(\d)", r"\1*\2", s)

    # Keep only safe math chars.
    if not re.fullmatch(r"[-+*/().0-9pi\s]+", s):
        return None

    try:
        return float(eval(s, {"__builtins__": {}}, {"pi": math.pi}))
    except Exception:
        return None

def same_answer(a: Optional[str], b: Optional[str], tol: float = 1e-9) -> bool:
    va, vb = parse_answer_value(a), parse_answer_value(b)

    if isinstance(va, tuple) and isinstance(vb, tuple) and len(va) == len(vb):
        return all(abs(x - y) <= tol for x, y in zip(va, vb))

    if isinstance(va, float) and isinstance(vb, float):
        return abs(va - vb) <= tol * max(1.0, abs(vb))

    return canonical_answer(a) == canonical_answer(b)

def relative_error(pred: Optional[str], gold: Optional[str]) -> Optional[float]:
    vp, vg = parse_answer_value(pred), parse_answer_value(gold)
    if isinstance(vp, float) and isinstance(vg, float):
        return abs(vp - vg) / max(1.0, abs(vg))
    if isinstance(vp, tuple) and isinstance(vg, tuple) and len(vp) == len(vg):
        # Local extension for coordinate answers.
        num = sum(abs(a - b) for a, b in zip(vp, vg))
        den = max(1.0, sum(abs(x) for x in vg))
        return num / den
    return 0.0 if same_answer(pred, gold) else None

def score_one(pred: Optional[str], gold: Optional[str]) -> int:
    err = relative_error(pred, gold)
    if err is None:
        return 0
    if err <= 0.01:
        return 10
    if err <= 0.10:
        return 5
    if err <= 0.50:
        return 1
    return 0

# Quick evaluator sanity check
_test_outputs = [
    "Lời giải ngắn: 48 - 11 = 37\nĐáp án là: 37",
    "abc #### 19",
    "Câu trả lời là: \\frac{9}{20}",
    "Đáp án là: 36\\pi",
]
for x in _test_outputs:
    print(x, "=>", extract_final_answer(x), "=>", parse_answer_value(extract_final_answer(x)))

Lời giải ngắn: 48 - 11 = 37
Đáp án là: 37 => 37 => 37.0
abc #### 19 => 19 => 19.0
Câu trả lời là: \frac{9}{20} => 20 => 20.0
Đáp án là: 36\pi => 36\pi => 113.09733552923255


# Cell 4 — Audit raw train/valid

In [4]:
# Cell 4 — Data audit functions

NOISY_PATTERNS = {
    "has_asy": r"\[asy\].*?\[/asy\]",
    "has_diagram_word": r"\b(diagram|sơ đồ|hình vẽ|asymptote)\b",
    "has_latex_error_dong_hop": r"đóng hộp|\\ đóng hộp",
    "has_duplicate_unknown_var_question": r"(Giá trị của biến [^\n?]+\?)\s*\1",
    "has_english_answer_anchor": r"The answer is",
    "has_old_vietnamese_anchor": r"Câu trả lời là",
}

def get_type(raw: Dict[str, Any]) -> str:
    return str(raw.get("type", "UNKNOWN") or "UNKNOWN")

def get_query(raw: Dict[str, Any]) -> str:
    return str(raw.get("query_vi", "") or "")

def get_response(raw: Dict[str, Any]) -> str:
    return str(raw.get("response_vi", "") or "")

def audit_records(raw_records: List[Dict[str, Any]], name: str, max_preview: int = 8) -> Dict[str, Any]:
    rows = []
    q_to_answers = defaultdict(list)

    for i, r in enumerate(raw_records):
        q = get_query(r)
        resp = get_response(r)
        ans = extract_final_answer(resp, fallback_last_number=False)
        qh = stable_hash(q)

        noisy_hits = {
            k: bool(re.search(pat, q + "\n" + resp, flags=re.IGNORECASE | re.DOTALL))
            for k, pat in NOISY_PATTERNS.items()
        }

        row = {
            "idx": i,
            "type": get_type(r),
            "query_len": len(q),
            "response_len": len(resp),
            "total_len": len(q) + len(resp),
            "missing_query": not q.strip(),
            "missing_response": not resp.strip(),
            "answer": ans,
            "answer_extract_ok": ans is not None and not is_bad_final_answer_candidate(ans),
            "has_dap_an_la": "Đáp án là" in resp,
            "has_multiple_anchors": sum(len(re.findall(p, resp, flags=re.IGNORECASE)) for p in ANSWER_PATTERNS) > 1,
            **noisy_hits,
        }
        rows.append(row)
        if row["answer_extract_ok"]:
            q_to_answers[qh].append(canonical_answer(ans))

    df = pd.DataFrame(rows)

    conflict_q = 0
    conflict_rows = 0
    for qh, answers in q_to_answers.items():
        if len(set(answers)) > 1:
            conflict_q += 1
            conflict_rows += len(answers)

    report = {
        "name": name,
        "count": len(raw_records),
        "missing_query": int(df["missing_query"].sum()) if len(df) else 0,
        "missing_response": int(df["missing_response"].sum()) if len(df) else 0,
        "answer_extract_success_rate": float(df["answer_extract_ok"].mean()) if len(df) else 0.0,
        "has_dap_an_la_rate": float(df["has_dap_an_la"].mean()) if len(df) else 0.0,
        "multiple_anchor_count": int(df["has_multiple_anchors"].sum()) if len(df) else 0,
        "conflict_query_count": conflict_q,
        "conflict_row_count": conflict_rows,
        "type_distribution": dict(Counter(df["type"])) if len(df) else {},
        "noisy_counts": {
            k: int(df[k].sum()) for k in NOISY_PATTERNS.keys()
        } if len(df) else {},
        "length_stats": {
            "query_p95": float(df["query_len"].quantile(0.95)) if len(df) else 0,
            "response_p95": float(df["response_len"].quantile(0.95)) if len(df) else 0,
            "total_p95": float(df["total_len"].quantile(0.95)) if len(df) else 0,
            "total_max": int(df["total_len"].max()) if len(df) else 0,
        }
    }

    print(f"\n===== AUDIT {name} =====")
    print(json.dumps(report, ensure_ascii=False, indent=2)[:5000])

    bad = df[~df["answer_extract_ok"]].head(max_preview)
    if len(bad):
        print(f"\n{name} examples with bad answer extraction:")
        display(bad[["idx", "type", "query_len", "response_len", "answer"]])

    return report

audit_train = audit_records(raw_train, "raw_train")
audit_valid = audit_records(raw_valid, "raw_valid")
write_json({"train": audit_train, "valid": audit_valid}, PROC_DIR / "raw_audit_report.json")


===== AUDIT raw_train =====
{
  "name": "raw_train",
  "count": 95400,
  "missing_query": 0,
  "missing_response": 0,
  "answer_extract_success_rate": 0.9537840670859539,
  "has_dap_an_la_rate": 0.7432704402515723,
  "multiple_anchor_count": 57879,
  "conflict_query_count": 56,
  "conflict_row_count": 228,
  "type_distribution": {
    "GSM_AnsAug": 18745,
    "MATH_AnsAug": 16999,
    "GSM_SV": 9869,
    "GSM_FOBAR": 10023,
    "MATH_Rephrased": 12477,
    "MATH_SV": 3591,
    "GSM_Rephrased": 20028,
    "MATH_FOBAR": 3668
  },
  "noisy_counts": {
    "has_asy": 878,
    "has_diagram_word": 268,
    "has_latex_error_dong_hop": 914,
    "has_duplicate_unknown_var_question": 3197,
    "has_english_answer_anchor": 270,
    "has_old_vietnamese_anchor": 25090
  },
  "length_stats": {
    "query_p95": 419.0,
    "response_p95": 980.0,
    "total_p95": 1327.050000000003,
    "total_max": 3821
  }
}

raw_train examples with bad answer extraction:


,idx,type,query_len,response_len,answer
22,22,MATH_AnsAug,353,250,None
53,53,MATH_SV,268,884,None
96,96,MATH_AnsAug,138,391,None
97,97,MATH_AnsAug,61,352,None
114,114,MATH_Rephrased,39,240,None
147,147,MATH_AnsAug,192,782,None
181,181,MATH_SV,216,1667,None
221,221,MATH_AnsAug,49,329,None



===== AUDIT raw_valid =====
{
  "name": "raw_valid",
  "count": 1000,
  "missing_query": 0,
  "missing_response": 0,
  "answer_extract_success_rate": 0.955,
  "has_dap_an_la_rate": 0.728,
  "multiple_anchor_count": 617,
  "conflict_query_count": 0,
  "conflict_row_count": 0,
  "type_distribution": {
    "GSM_Rephrased": 197,
    "MATH_Rephrased": 116,
    "MATH_SV": 41,
    "GSM_AnsAug": 209,
    "GSM_SV": 97,
    "GSM_FOBAR": 122,
    "MATH_AnsAug": 173,
    "MATH_FOBAR": 45
  },
  "noisy_counts": {
    "has_asy": 7,
    "has_diagram_word": 1,
    "has_latex_error_dong_hop": 7,
    "has_duplicate_unknown_var_question": 37,
    "has_english_answer_anchor": 2,
    "has_old_vietnamese_anchor": 271
  },
  "length_stats": {
    "query_p95": 443.04999999999995,
    "response_p95": 1081.2999999999997,
    "total_p95": 1424.6999999999994,
    "total_max": 2667
  }
}

raw_valid examples with bad answer extraction:


,idx,type,query_len,response_len,answer
18,18,MATH_AnsAug,219,323,None
31,31,MATH_Rephrased,66,208,None
37,37,MATH_AnsAug,154,327,None
42,42,MATH_Rephrased,162,482,None
102,102,MATH_AnsAug,50,310,None
104,104,MATH_AnsAug,125,830,None
123,123,MATH_AnsAug,124,640,None
163,163,MATH_Rephrased,87,389,None


# Cell 5 — Cleaning + full-reasoning target normalization

Variant 9 không rút target xuống 1–2 câu đầu nữa. Thay vào đó, giữ phần lời giải gốc sau khi bỏ các anchor cũ, cắt an toàn theo ký tự, rồi gắn một anchor cuối duy nhất `Đáp án là:`.

In [5]:
# Cell 5 — Cleaning + full-reasoning target normalization

def fix_artifacts(text: str) -> str:
    text = normalize_unicode_text(text)
    text = text.replace("\\time", "\\times")
    text = text.replace("\\dfrac", "\\frac")
    text = text.replace("\\ đóng hộp", "\\boxed")
    text = text.replace("\\đóng hộp", "\\boxed")
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def strip_asy_blocks(text: str) -> str:
    return re.sub(r"\[asy\].*?\[/asy\]", " ", str(text or ""), flags=re.IGNORECASE | re.DOTALL)

def normalize_decimal_commas_in_text(text: str) -> str:
    # 4,5 -> 4.5 but keep 1,000 as thousand separator if present.
    return re.sub(r"(?<!\d)(\d+),(\d{1,2})(?!\d)", lambda m: f"{m.group(1)}.{m.group(2)}", text)

def remove_old_answer_tails(text: str) -> str:
    text = normalize_unicode_text(text)
    # Remove from the earliest known final answer anchor to avoid duplicated answer tails.
    spans = []
    for pat in ANSWER_PATTERNS:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            spans.append(m.span()[0])
    if spans:
        text = text[:min(spans)].strip()
    return text.strip()

def split_sentences_vi(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    parts = re.split(r"(?<=[.!?。])\s+", text)
    return [p.strip() for p in parts if p.strip()]

def keep_original_reasoning(response: str, answer: str, max_chars: int = 700) -> str:
    """
    Variant 9 target quality:
    - Giữ lời giải gốc để model học multi-step reasoning.
    - Chỉ bỏ các anchor cũ kiểu ####, Câu trả lời là, The answer is, Đáp án là.
    - Cắt an toàn theo ký tự để vẫn vừa MAX_LENGTH.
    """
    body = remove_old_answer_tails(response)
    body = strip_boxed(body)
    body = re.sub(r"#+\s*[-+]?\d+(?:[\.,]\d+)?\s*$", "", body).strip()
    body = re.sub(r"\s+", " ", body).strip()

    if len(body) > max_chars:
        # Ưu tiên cắt ở cuối câu gần max_chars.
        cut = body[:max_chars]
        sentence_cut = max(cut.rfind("."), cut.rfind("?"), cut.rfind("!"))
        if sentence_cut >= int(max_chars * 0.55):
            body = cut[:sentence_cut + 1].strip()
        else:
            body = cut.rsplit(" ", 1)[0].strip()

    if not body:
        body = "Tính theo dữ kiện trong đề."

    return body

def normalize_target(response: str, final_answer: str) -> str:
    body = keep_original_reasoning(response, final_answer, max_chars=700)
    final_answer = clean_answer_candidate(final_answer)
    return f"Lời giải: {body}\nĐáp án là: {final_answer}"

def make_prompt(query: str) -> str:
    query = re.sub(r"\s+", " ", fix_artifacts(strip_asy_blocks(query))).strip()
    return f"Câu hỏi: {query}\n\nLời giải:"

def normalize_query(query: str) -> str:
    query = fix_artifacts(query)
    query = strip_asy_blocks(query)
    query = normalize_decimal_commas_in_text(query)
    query = re.sub(r"\s+", " ", query).strip()
    return query

def normalize_response(response: str) -> str:
    response = fix_artifacts(response)
    response = strip_asy_blocks(response)
    response = normalize_decimal_commas_in_text(response)
    response = re.sub(r"\s+", " ", response).strip()
    return response

def should_drop_record(query: str, response: str, answer: Optional[str]) -> Tuple[bool, str]:
    if not query.strip():
        return True, "empty_query"
    if not response.strip():
        return True, "empty_response"
    if answer is None:
        return True, "answer_extract_failed"
    if is_bad_final_answer_candidate(answer):
        return True, "bad_final_answer"
    if len(query) > MAX_QUERY_CHARS:
        return True, "query_too_long"
    if len(response) > MAX_RESPONSE_CHARS:
        return True, "response_too_long"
    if len(query) + len(response) > MAX_TOTAL_CHARS:
        return True, "total_too_long"
    if re.search(r"ble x is|ble x là", response, flags=re.IGNORECASE):
        return True, "broken_translation_ble_x"
    return False, ""

def preprocess_split(raw_records: List[Dict[str, Any]], split_name: str, train_mode: bool) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    processed = []
    drop_logs = []

    for idx, raw in enumerate(raw_records):
        q_raw = get_query(raw)
        r_raw = get_response(raw)

        q = normalize_query(q_raw)
        r = normalize_response(r_raw)

        ans = extract_final_answer(r, fallback_last_number=False)
        drop, reason = should_drop_record(q, r, ans)

        if drop:
            drop_logs.append({
                "split": split_name,
                "idx": idx,
                "type": get_type(raw),
                "reason": reason,
                "query_preview": q[:250],
                "response_tail": r[-250:],
                "answer": ans,
            })
            continue

        target = normalize_target(r, ans)

        original_question = str(raw.get("original_question_vi", "") or q)
        rec = {
            "id": raw.get("id", idx),
            "raw_index": idx,
            "type": get_type(raw),
            "query_vi": q,
            "response_vi_original": r,
            "response_vi": target,
            "final_answer": clean_answer_candidate(ans),
            "prompt": make_prompt(q),
            "source_group": get_type(raw).split("_", 1)[0] if "_" in get_type(raw) else get_type(raw),
            "aug_type": get_type(raw).split("_", 1)[1] if "_" in get_type(raw) else "UNKNOWN",
            "query_hash": stable_hash(q),
            "original_question_hash": stable_hash(original_question),
            "query_response_hash": stable_hash(q + "\n" + target),
            "answer_conflict_key": canonical_answer(ans),
        }
        processed.append(rec)

    if train_mode:
        # Exact query+response dedup.
        seen = set()
        deduped = []
        for r in processed:
            key = r["query_response_hash"]
            if key in seen:
                drop_logs.append({
                    "split": split_name,
                    "idx": r["raw_index"],
                    "type": r["type"],
                    "reason": "exact_duplicate",
                    "query_preview": r["query_vi"][:250],
                    "answer": r["final_answer"],
                })
                continue
            seen.add(key)
            deduped.append(r)
        processed = deduped

        # Query-level conflicts: same query, different answer => drop all conflicted rows.
        q_to_answers = defaultdict(set)
        for r in processed:
            q_to_answers[r["query_hash"]].add(r["answer_conflict_key"])

        conflicted_q = {q for q, answers in q_to_answers.items() if len(answers) > 1}
        if conflicted_q:
            kept = []
            for r in processed:
                if r["query_hash"] in conflicted_q:
                    drop_logs.append({
                        "split": split_name,
                        "idx": r["raw_index"],
                        "type": r["type"],
                        "reason": "query_conflict_drop_all",
                        "query_preview": r["query_vi"][:250],
                        "answer": r["final_answer"],
                    })
                else:
                    kept.append(r)
            processed = kept

        # Keep clean subset, balanced by type.
        if TRAIN_MAX_SAMPLES is not None and len(processed) > TRAIN_MAX_SAMPLES:
            rng = random.Random(SEED)
            by_type = defaultdict(list)
            for r in processed:
                by_type[r["type"]].append(r)

            target_per_type = max(1, TRAIN_MAX_SAMPLES // max(1, len(by_type)))
            selected = []
            remaining_pool = []

            for t, items in by_type.items():
                rng.shuffle(items)
                selected.extend(items[:target_per_type])
                remaining_pool.extend(items[target_per_type:])

            if len(selected) < TRAIN_MAX_SAMPLES:
                rng.shuffle(remaining_pool)
                selected.extend(remaining_pool[:TRAIN_MAX_SAMPLES - len(selected)])

            rng.shuffle(selected)
            processed = selected[:TRAIN_MAX_SAMPLES]

    return processed, drop_logs


# Cell 6 — Run preprocessing and export

In [6]:
# Cell 6 — Run preprocessing and export clean train/valid

train_clean, train_drop_logs = preprocess_split(raw_train, "train", train_mode=True)
valid_clean, valid_drop_logs = preprocess_split(raw_valid, "valid", train_mode=False)

write_jsonl(train_clean, PROC_DIR / "train_clean.jsonl")
write_jsonl(valid_clean, PROC_DIR / "valid_clean.jsonl")
write_jsonl(train_drop_logs + valid_drop_logs, PROC_DIR / "drop_log.jsonl")

preprocess_report = {
    "variant_name": VARIANT_NAME,
    "config": {
        "TRAIN_MAX_SAMPLES": TRAIN_MAX_SAMPLES,
        "MAX_QUERY_CHARS": MAX_QUERY_CHARS,
        "MAX_RESPONSE_CHARS": MAX_RESPONSE_CHARS,
        "MAX_TOTAL_CHARS": MAX_TOTAL_CHARS,
        "target_format": "Lời giải: <full original reasoning>\\nĐáp án là: {answer}",
    },
    "counts": {
        "raw_train": len(raw_train),
        "raw_valid": len(raw_valid),
        "train_clean": len(train_clean),
        "valid_clean": len(valid_clean),
        "train_dropped": len(train_drop_logs),
        "valid_dropped": len(valid_drop_logs),
    },
    "drop_summary": dict(Counter(x["reason"] for x in train_drop_logs + valid_drop_logs)),
    "train_type_distribution": dict(Counter(x["type"] for x in train_clean)),
    "valid_type_distribution": dict(Counter(x["type"] for x in valid_clean)),
}

write_json(preprocess_report, PROC_DIR / "preprocess_report.json")
print(json.dumps(preprocess_report, ensure_ascii=False, indent=2)[:8000])

print("\nSample processed target:")
print(json.dumps(train_clean[0], ensure_ascii=False, indent=2)[:1500])

{
  "config": {
    "TRAIN_MAX_SAMPLES": 50000,
    "MAX_QUERY_CHARS": 900,
    "MAX_RESPONSE_CHARS": 1400,
    "MAX_TOTAL_CHARS": 1900,
    "target_format": "Lời giải ngắn: ...\\nĐáp án là: {answer}"
  },
  "counts": {
    "raw_train": 95400,
    "raw_valid": 1000,
    "train_clean": 50000,
    "valid_clean": 937,
    "train_dropped": 7923,
    "valid_dropped": 63
  },
  "drop_summary": {
    "answer_extract_failed": 4455,
    "response_too_long": 1094,
    "total_too_long": 46,
    "broken_translation_ble_x": 36,
    "query_too_long": 52,
    "exact_duplicate": 2053,
    "query_conflict_drop_all": 250
  },
  "train_type_distribution": {
    "MATH_Rephrased": 6908,
    "GSM_SV": 6753,
    "MATH_AnsAug": 7374,
    "GSM_Rephrased": 8226,
    "GSM_AnsAug": 7941,
    "MATH_SV": 3122,
    "GSM_FOBAR": 6753,
    "MATH_FOBAR": 2923
  },
  "valid_type_distribution": {
    "GSM_Rephrased": 197,
    "MATH_Rephrased": 93,
    "MATH_SV": 36,
    "GSM_AnsAug": 209,
    "GSM_SV": 94,
    "GSM_FOBAR

# Cell 7 — Load tokenizer/model offline

In [7]:
# Cell 7 — Load tokenizer/model offline

from transformers import AutoTokenizer, AutoModelForCausalLM

def find_model_path() -> Path:
    for p in MODEL_PATH_CANDIDATES:
        if p.exists():
            # Direct model dir
            if (p / "config.json").exists():
                return p
            # Search inside Kaggle dataset folder
            for sub in p.rglob("config.json"):
                return sub.parent

    # Last fallback: search all /kaggle/input for a config that looks like GPT-2
    root = Path("/kaggle/input")
    for cfg in root.rglob("config.json"):
        try:
            data = json.loads(cfg.read_text(encoding="utf-8"))
            mt = str(data.get("model_type", "")).lower()
            if "gpt2" in mt:
                return cfg.parent
        except Exception:
            pass

    raise FileNotFoundError(
        "Cannot find local GPT-2 Vietnamese model. "
        "Please add the NlpHUST/gpt2-vietnamese Kaggle dataset and update MODEL_PATH_CANDIDATES."
    )

MODEL_PATH = find_model_path()
print("Using MODEL_PATH:", MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)

tokenizer.eos_token_id = SAFE_EOS_ID
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = SAFE_EOS_ID

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID
model.config.use_cache = False

print("Tokenizer vocab size:", len(tokenizer))
print("pad_token_id:", tokenizer.pad_token_id, "eos_token_id:", model.config.eos_token_id)
print("Model parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

Using MODEL_PATH: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer vocab size: 50258
pad_token_id: 50256 eos_token_id: 50256
Model parameters: 124.439808 M


# Cell 8 — Build torch Dataset with prompt masking

In [8]:
# Cell 8 — Build torch Dataset with prompt masking

from torch.utils.data import Dataset

class MathCausalDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]], tokenizer, max_length: int):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        prompt = r["prompt"]
        target = r["response_vi"].strip() + tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )["input_ids"]

        full = self.tokenizer(
            prompt + " " + target,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )

        input_ids = full["input_ids"]
        attention_mask = full["attention_mask"]
        labels = input_ids.copy()

        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

class DataCollatorForCausalLMWithPadding:
    def __init__(self, tokenizer, label_pad_token_id: int = -100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
            batch["labels"].append(f["labels"] + [self.label_pad_token_id] * pad_len)

        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

# Small eval subset during training to save time; final evaluation is done later on full valid_clean.
rng = random.Random(SEED)
train_records = train_clean.copy()
rng.shuffle(train_records)

eval_during_train = valid_clean.copy()
if len(eval_during_train) > 512:
    eval_during_train = rng.sample(eval_during_train, 512)

train_dataset = MathCausalDataset(train_records, tokenizer, MAX_LENGTH)
eval_dataset = MathCausalDataset(eval_during_train, tokenizer, MAX_LENGTH)
collator = DataCollatorForCausalLMWithPadding(tokenizer)

print("train_dataset:", len(train_dataset))
print("eval_dataset during train:", len(eval_dataset))

train_dataset: 50000
eval_dataset during train: 512


# Cell 9 — Apply LoRA

In [9]:
# Cell 9 — Apply LoRA

try:
    from peft import LoraConfig, get_peft_model, TaskType
except Exception as e:
    raise ImportError(
        "PEFT is required for LoRA. Kaggle usually includes peft. "
        "If this fails, add a Kaggle dataset/package that contains peft, still with Internet OFF."
    ) from e

# GPT-2 modules usually include c_attn/c_proj/c_fc.
target_modules = ["c_attn", "c_proj", "c_fc"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=target_modules,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    model = model.cuda()

trainable params: 1,179,648 || all params: 125,619,456 || trainable%: 0.9391


# Cell 10 — Fine-tune LoRA

In [10]:
from transformers import Trainer, TrainingArguments

def build_training_args():
    raw_kwargs = dict(
        output_dir=str(MODEL_OUT_DIR),
        overwrite_output_dir=True,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_steps=EVAL_STEPS,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
        dataloader_num_workers=2,
        remove_unused_columns=False,
        load_best_model_at_end=False,
    )

    sig = inspect.signature(TrainingArguments.__init__).parameters

    if "eval_strategy" in sig:
        raw_kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in sig:
        raw_kwargs["evaluation_strategy"] = "steps"

    # Chỉ giữ những argument mà version transformers hiện tại hỗ trợ.
    kwargs = {k: v for k, v in raw_kwargs.items() if k in sig}

    dropped = sorted(set(raw_kwargs.keys()) - set(kwargs.keys()))
    if dropped:
        print("Dropped unsupported TrainingArguments:", dropped)

    return TrainingArguments(**kwargs)

def build_trainer_kwargs(training_args: TrainingArguments) -> Dict[str, Any]:
    base = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collator,
    )

    sig = inspect.signature(Trainer.__init__).parameters
    if "tokenizer" in sig:
        base["tokenizer"] = tokenizer
    elif "processing_class" in sig:
        base["processing_class"] = tokenizer

    return base


training_args = build_training_args()
trainer = Trainer(**build_trainer_kwargs(training_args))

train_result = trainer.train()

trainer.save_model(str(MODEL_OUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUT_DIR))

write_json({
    "variant_name": VARIANT_NAME,
    "train_result": train_result.metrics,
    "training_config": {
        "TRAIN_MAX_SAMPLES": TRAIN_MAX_SAMPLES,
        "MAX_LENGTH": MAX_LENGTH,
        "PER_DEVICE_TRAIN_BATCH_SIZE": PER_DEVICE_TRAIN_BATCH_SIZE,
        "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
        "LEARNING_RATE": LEARNING_RATE,
        "NUM_TRAIN_EPOCHS": NUM_TRAIN_EPOCHS,
        "LORA_R": LORA_R,
        "LORA_ALPHA": LORA_ALPHA,
    }
}, MODEL_OUT_DIR / "train_metrics.json")

print(train_result.metrics)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 50257, 'pad_token_id': 50256}.


Dropped unsupported TrainingArguments: ['overwrite_output_dir']


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
250,1.634123,1.586844
500,1.484336,1.475671
750,1.477143,1.449060


{'train_runtime': 1196.8512, 'train_samples_per_second': 41.776, 'train_steps_per_second': 0.653, 'total_flos': 5660872194097152.0, 'train_loss': 1.6442349560730292, 'epoch': 1.0}


# Cell 11 — 5-candidate generation + true majority voting verifier

Variant 9 sinh 5 ứng viên: greedy, beam5, sample t=0.7, sample t=0.9, top_k=50. Nếu có majority thật sự từ ít nhất 2 ứng viên, chọn nhóm majority; nếu không thì fallback về verifier score.

In [11]:
# Cell 11 — 5-candidate generation + true majority voting verifier

model.eval()

def normalize_generated_text(text: str) -> str:
    text = normalize_unicode_text(text)
    text = text.replace(tokenizer.eos_token or "", "")
    text = re.sub(r"\s+", " ", text).strip()

    # Remove repeated prompt fragments if any.
    text = re.sub(r"^Câu hỏi:.*?Lời giải\s*[:：]?\s*", "", text, flags=re.IGNORECASE | re.DOTALL).strip()
    text = re.sub(r"^Câu hỏi:.*?Lời giải ngắn\s*[:：]?\s*", "", text, flags=re.IGNORECASE | re.DOTALL).strip()

    # Ensure visible output starts with a stable solution prefix.
    if not text.lower().startswith("lời giải"):
        text = "Lời giải: " + text

    # Normalize old anchors to strict anchor.
    text = re.sub(r"Câu trả lời là\s*[:：]?", "Đáp án là:", text, flags=re.IGNORECASE)
    text = re.sub(r"The answer is\s*[:：]?", "Đáp án là:", text, flags=re.IGNORECASE)
    text = re.sub(r"####\s*", "Đáp án là: ", text)

    return text.strip()

def force_final_format(text: str, answer: Optional[str]) -> str:
    text = normalize_generated_text(text)
    answer = clean_answer_candidate(answer or extract_final_answer(text) or "")

    body = remove_old_answer_tails(text).strip()
    body = re.sub(r"^(Lời giải ngắn|Lời giải)\s*[:：]?\s*", "", body, flags=re.IGNORECASE).strip()
    if not body:
        body = "Tính theo dữ kiện trong đề."

    # Variant 9 allows a longer explanation than variant 6, but keeps output bounded.
    if len(body) > 800:
        body = body[:800].rsplit(" ", 1)[0].strip()

    return f"Lời giải: {body}\nĐáp án là: {answer}"

def repetition_penalty_score(text: str) -> int:
    toks = text.lower().split()
    if len(toks) < 8:
        return 0
    bigrams = list(zip(toks, toks[1:]))
    if not bigrams:
        return 0
    unique_ratio = len(set(bigrams)) / len(bigrams)
    if unique_ratio < 0.55:
        return -8
    if unique_ratio < 0.70:
        return -3
    return 2

def final_equation_matches_answer(text: str, answer: Optional[str]) -> bool:
    if answer is None:
        return False
    ans_val = parse_answer_value(answer)
    if not isinstance(ans_val, (int, float)):
        return False

    eqs = re.findall(
        r"([-+]?\d+(?:\.\d+)?(?:\s*[\+\-\*/]\s*[-+]?\d+(?:\.\d+)?){1,5})\s*=\s*([-+]?\d+(?:\.\d+)?)",
        text,
    )
    for lhs, rhs in eqs[-4:]:
        try:
            lhs_val = float(eval(lhs, {"__builtins__": {}}, {}))
            rhs_val = float(rhs)
            if abs(lhs_val - rhs_val) <= 1e-9 and abs(rhs_val - float(ans_val)) <= 1e-9:
                return True
        except Exception:
            continue
    return False

def answer_appears_before_anchor(text: str, answer: Optional[str]) -> bool:
    if not answer:
        return False
    body = remove_old_answer_tails(text)
    nums = re.findall(r"[-+]?\d+(?:[\.,]\d+)?(?:\s*/\s*[-+]?\d+(?:[\.,]\d+)?)?", body)
    if not nums:
        return False
    # Prefer the final numbers immediately before the answer anchor.
    for x in nums[-4:]:
        if same_answer(clean_answer_candidate(x), answer):
            return True
    return False

def score_candidate(text: str, majority_answer: Optional[str] = None) -> Dict[str, Any]:
    norm = normalize_generated_text(text)
    ans = extract_final_answer(norm, fallback_last_number=True)

    score = 0
    reasons = []

    if "Đáp án là" in norm:
        score += 5
        reasons.append("has_anchor")
    else:
        score -= 5
        reasons.append("missing_anchor")

    if ans is not None:
        score += 10
        reasons.append("extractable")
    else:
        score -= 100
        reasons.append("not_extractable")

    if ans is not None and not is_bad_final_answer_candidate(ans):
        score += 10
        reasons.append("valid_answer")
    else:
        score -= 30
        reasons.append("bad_answer")

    if len(norm) <= 980:
        score += 3
        reasons.append("bounded_length")
    else:
        score -= 4
        reasons.append("too_long")

    rep_score = repetition_penalty_score(norm)
    score += rep_score
    reasons.append("no_heavy_repetition" if rep_score >= 0 else "repetition")

    if final_equation_matches_answer(norm, ans):
        score += 7
        reasons.append("equation_matches_answer")

    if answer_appears_before_anchor(norm, ans):
        score += 3
        reasons.append("answer_appears_before_anchor")

    if majority_answer is not None and same_answer(ans, majority_answer):
        score += 12
        reasons.append("true_majority_answer")

    return {
        "text": norm,
        "answer": ans,
        "score": score,
        "reasons": reasons,
    }

def choose_majority_answer(candidates: List[Dict[str, Any]], min_votes: int = 2) -> Optional[str]:
    valid_answers = [
        c["answer"] for c in candidates
        if c.get("answer") is not None and not is_bad_final_answer_candidate(c.get("answer"))
    ]
    if not valid_answers:
        return None

    groups = []
    for ans in valid_answers:
        placed = False
        for g in groups:
            if same_answer(ans, g[0]):
                g.append(ans)
                placed = True
                break
        if not placed:
            groups.append([ans])

    groups.sort(key=len, reverse=True)
    if groups and len(groups[0]) >= min_votes:
        return groups[0][0]
    return None

def select_best_output(candidate_texts: List[str]) -> Tuple[str, Dict[str, Any]]:
    first_pass = [score_candidate(t) for t in candidate_texts]
    majority = choose_majority_answer(first_pass, min_votes=2)

    second_pass = [score_candidate(t, majority_answer=majority) for t in candidate_texts]

    if majority is not None:
        group_candidates = [c for c in second_pass if same_answer(c.get("answer"), majority)]
        best = max(group_candidates, key=lambda x: x["score"])
        voting = "majority"
    else:
        best = max(second_pass, key=lambda x: x["score"])
        voting = "verifier_fallback"

    final_text = force_final_format(best["text"], majority if majority is not None else best["answer"])

    best_info = {
        "selected_answer": clean_answer_candidate((majority if majority is not None else best["answer"]) or ""),
        "selected_score": best["score"],
        "selected_reasons": best["reasons"],
        "majority_answer": clean_answer_candidate(majority or ""),
        "voting": voting,
        "all_candidates": sorted(second_pass, key=lambda x: x["score"], reverse=True),
    }
    return final_text, best_info

GEN_CONFIGS = [
    {"name": "greedy", "do_sample": False, "num_beams": 1},
    {"name": "beam5", "do_sample": False, "num_beams": 5},
    {"name": "sample_t07", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "num_beams": 1},
    {"name": "sample_t09", "do_sample": True, "temperature": 0.9, "top_p": 0.95, "num_beams": 1},
    {"name": "topk50", "do_sample": True, "top_k": 50, "top_p": 0.9, "temperature": 0.8, "num_beams": 1},
]

@torch.no_grad()
def generate_candidates_for_prompt(prompt: str) -> List[str]:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    outputs = []

    for cfg in GEN_CONFIGS:
        gen_kwargs = dict(
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
            repetition_penalty=1.10,
            no_repeat_ngram_size=3,
        )
        gen_kwargs.update({k: v for k, v in cfg.items() if k != "name"})

        out_ids = model.generate(**inputs, **gen_kwargs)
        decoded = tokenizer.decode(out_ids[0], skip_special_tokens=True)

        if decoded.startswith(prompt):
            tail = decoded[len(prompt):].strip()
            decoded = "Lời giải: " + tail

        outputs.append(decoded)

    return outputs

# Smoke test on 2 validation samples
for r in valid_clean[:2]:
    cands = generate_candidates_for_prompt(r["prompt"])
    selected, info = select_best_output(cands)
    print("\nQUERY:", r["query_vi"][:200])
    print("GOLD:", r["final_answer"])
    print("SELECTED:", selected)
    print("INFO:", {k: v for k, v in info.items() if k != "all_candidates"})



QUERY: Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô
GOLD: 37
SELECTED: Lời giải ngắn: Lời giải ngắn x phút: Susan đang ở trong một trò cờ bàn, vì vậy cô ấy đang ở giữa hai ô bắt buộc. Cô ấy đang chơi trò cờ Bàn có 48 - 48 = 48 ô.
Đáp án là: 4
INFO: {'selected_answer': '4', 'selected_score': 38, 'selected_reasons': ['has_anchor', 'extractable', 'valid_answer', 'short_enough', 'no_heavy_repetition', 'majority_answer'], 'majority_answer': '4'}

QUERY: Nếu $\angle PQR = \angle PRQ$, và độ dài của QR và PR lần lượt là 5 và 7 thì chu vi của tam giác PQR là bao nhiêu?
GOLD: 19
SELECTED: Lời giải ngắn: Lời giải ngắn : Độ dài của hình chữ nhật là $5$ và độ rộng của hình vuông là $7$. Độ dài và chiều rộng của tam giac là $3$.
Đáp án là: 3
INFO: {'selected_answer': '3', 'selected_score': 38, 'selected_reasons': ['has_anchor', 'extractable', '

# Cell 12 — Full validation inference + scoring

In [12]:
# Cell 12 — Full validation inference + scoring

def evaluate_records(records: List[Dict[str, Any]], max_samples: Optional[int] = None) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    eval_records = records
    if max_samples is not None and len(records) > max_samples:
        rng = random.Random(SEED)
        eval_records = rng.sample(records, max_samples)

    outputs = []

    for i, r in enumerate(eval_records):
        if i % 50 == 0:
            print(f"Evaluating {i}/{len(eval_records)}")

        candidate_texts = generate_candidates_for_prompt(r["prompt"])
        selected_text, verifier_info = select_best_output(candidate_texts)
        pred_answer = extract_final_answer(selected_text, fallback_last_number=True)
        gold_answer = r["final_answer"]

        err = relative_error(pred_answer, gold_answer)
        ex_score = score_one(pred_answer, gold_answer)

        outputs.append({
            "id": r.get("id", i),
            "raw_index": r.get("raw_index", i),
            "query_vi": r["query_vi"],
            "type": r["type"],
            "gold_answer": gold_answer,
            "pred_answer": clean_answer_candidate(pred_answer or ""),
            "relative_error": err,
            "score": ex_score,
            "model_output": selected_text,
            "verifier": {
                "selected_score": verifier_info["selected_score"],
                "selected_reasons": verifier_info["selected_reasons"],
                "majority_answer": verifier_info["majority_answer"],
                "voting": verifier_info.get("voting", ""),
            },
            "candidate_outputs": candidate_texts,
        })

    total = len(outputs)
    raw_score = sum(x["score"] for x in outputs)
    report = {
        "num_examples": total,
        "raw_score": raw_score,
        "score_10": raw_score / total if total else 0.0,
        "extract_rate": sum(bool(x["pred_answer"]) for x in outputs) / total if total else 0.0,
        "score_distribution": dict(Counter(x["score"] for x in outputs)),
        "avg_relative_error_parseable": float(np.mean([x["relative_error"] for x in outputs if x["relative_error"] is not None])) if any(x["relative_error"] is not None for x in outputs) else None,
    }

    return outputs, report

valid_outputs, valid_report = evaluate_records(valid_clean, max_samples=VALID_MAX_SAMPLES)

write_json(valid_outputs, PRED_DIR / "valid_output_detailed.json")
write_json(valid_report, PRED_DIR / "valid_report.json")

# Required simpler valid_output format
valid_output_simple = [
    {
        "id": x["id"],
        "query_vi": x["query_vi"],
        "type": x["type"],
        "model_output": x["model_output"],
    }
    for x in valid_outputs
]
write_json(valid_output_simple, WORK_DIR / "valid_output.json")
write_json(valid_report, WORK_DIR / "valid_report.json")

print(json.dumps(valid_report, ensure_ascii=False, indent=2))

Evaluating 0/937
Evaluating 50/937
Evaluating 100/937
Evaluating 150/937
Evaluating 200/937
Evaluating 250/937
Evaluating 300/937
Evaluating 350/937
Evaluating 400/937
Evaluating 450/937
Evaluating 500/937
Evaluating 550/937
Evaluating 600/937
Evaluating 650/937
Evaluating 700/937
Evaluating 750/937
Evaluating 800/937
Evaluating 850/937
Evaluating 900/937
{
  "num_examples": 937,
  "raw_score": 676,
  "score_10": 0.7214514407684098,
  "extract_rate": 1.0,
  "score_distribution": {
    "0": 737,
    "1": 141,
    "5": 11,
    "10": 48
  },
  "avg_relative_error_parseable": 19.518398935409714
}


# Cell 13 — Report by type + error tables

In [13]:
# Cell 13 — Evaluation report by type + error tables

df_eval = pd.DataFrame(valid_outputs)

def summarize_group(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n": len(g),
        "score_10": g["score"].sum() / len(g) if len(g) else 0,
        "extract_rate": (g["pred_answer"].astype(str).str.len() > 0).mean() if len(g) else 0,
        "score_10_count": int((g["score"] == 10).sum()),
        "score_5_count": int((g["score"] == 5).sum()),
        "score_1_count": int((g["score"] == 1).sum()),
        "score_0_count": int((g["score"] == 0).sum()),
        "avg_relative_error_parseable": g["relative_error"].dropna().mean() if g["relative_error"].notna().any() else np.nan,
    })

type_report = df_eval.groupby("type").apply(summarize_group).reset_index()
type_report = type_report.sort_values(["score_10", "n"], ascending=[True, False])

display(type_report)
type_report.to_csv(WORK_DIR / "valid_report_by_type.csv", index=False, encoding="utf-8-sig")

# Worst examples for debugging
debug_cols = ["raw_index", "type", "score", "relative_error", "gold_answer", "pred_answer", "query_vi", "model_output"]
errors_df = df_eval[df_eval["score"] < 10].sort_values(["score", "relative_error"], ascending=[True, False], na_position="last")
display(errors_df[debug_cols].head(30))
errors_df[debug_cols].to_csv(WORK_DIR / "valid_errors.csv", index=False, encoding="utf-8-sig")

full_summary = {
    "overall": valid_report,
    "by_type": type_report.to_dict(orient="records"),
    "preprocess_report": preprocess_report,
}
write_json(full_summary, WORK_DIR / "full_validation_summary.json")

print("Saved:")
print("- /kaggle/working/valid_output.json")
print("- /kaggle/working/valid_report.json")
print("- /kaggle/working/valid_report_by_type.csv")
print("- /kaggle/working/valid_errors.csv")
print("- /kaggle/working/full_validation_summary.json")


,type,n,score_10,extract_rate,score_10_count,score_5_count,score_1_count,score_0_count,avg_relative_error_parseable
2,GSM_Rephrased,197.0,0.340102,1.0,3.0,3.0,22.0,169.0,1.717605
6,MATH_Rephrased,93.0,0.516129,1.0,3.0,1.0,13.0,76.0,53.240584
0,GSM_AnsAug,209.0,0.593301,1.0,7.0,5.0,29.0,168.0,2.412629
3,GSM_SV,94.0,0.712766,1.0,4.0,1.0,22.0,67.0,87.574772
4,MATH_AnsAug,151.0,0.715232,1.0,9.0,0.0,18.0,124.0,25.336608
1,GSM_FOBAR,121.0,1.280992,1.0,13.0,0.0,25.0,83.0,4.427943
5,MATH_FOBAR,36.0,1.305556,1.0,4.0,0.0,7.0,25.0,2.056108
7,MATH_SV,36.0,1.666667,1.0,5.0,1.0,5.0,25.0,1.196892


,raw_index,type,score,relative_error,gold_answer,pred_answer,query_vi,model_output
791,845,GSM_SV,0,7999.000000,2,16.000,Dakota bị xe buýt đâm và phải nằm viện 3 ngày....,Lời giải ngắn: Lời giải ngắn hạn: Hãy chia nhỏ...
661,707,MATH_AnsAug,0,3349.000000,3,10050,Hỏi tổng của 102 số đếm đầu tiên chia cho 5250...,"Lời giải ngắn: Lời giải ngắn, chúng ta có thể ..."
152,160,MATH_Rephrased,0,2246.666667,3,6743,Xác định ước chung lớn nhất của 654321 và 543210.,Lời giải ngắn: Lời giải ngắn là: $654321 = 673...
893,955,MATH_Rephrased,0,2246.666667,3,6743,Xác định ước chung lớn nhất của 654321 và 543210.,Lời giải ngắn: Lời giải ngắn là: $654321 = 673...
922,985,GSM_FOBAR,0,249.000000,2,500,Một máy bay phản lực bay 580 dặm trong x giờ. ...,Lời giải ngắn: Lời giải ngắn : Chúng ta biết r...
866,927,GSM_AnsAug,0,132.333333,6,800,Brett có nhiều hơn 24 viên bi xanh so với số v...,Lời giải ngắn: Lời giải ngắn x = 4x + 6 = 8x +...
63,68,MATH_AnsAug,0,99.000000,10,1000,"Một người bán tạp hóa trưng bày các lon, trong...",Lời giải ngắn: Lời giải ngắn x Đọc số hàng hiể...
253,268,GSM_SV,0,99.000000,5,500,Một cửa hàng kẹo sử dụng màu thực phẩm trong n...,Lời giải ngắn: Lời giải ngắn hạn: Hãy chia nhỏ...
642,688,GSM_FOBAR,0,49.000000,2,100,Một bà nội trợ đi chợ. Cô ấy đã tiêu x trong s...,Lời giải ngắn: Lời giải ngắn hạn: Bà nội trợ đ...
573,612,GSM_Rephrased,0,39.000000,50,2000,Nếu Silvia muốn mua một cây đàn guitar mới trự...,Lời giải ngắn: Lời giải ngắn x1 = $1000$ cho m...


Saved:
- /kaggle/working/valid_output.json
- /kaggle/working/valid_report.json
- /kaggle/working/valid_report_by_type.csv
- /kaggle/working/valid_errors.csv
- /kaggle/working/full_validation_summary.json


# Cell 14 — Generate test_predictions.json when test.json is available

In [14]:
def build_test_records(raw_test: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    records = []

    for i, raw in enumerate(raw_test):
        q = normalize_query(str(raw.get("query_vi", "") or ""))
        t = str(raw.get("type", "UNKNOWN") or "UNKNOWN")

        records.append({
            "id": raw.get("id", i),
            "query_vi": q,
            "type": t,
            "prompt": make_prompt(q),
        })

    return records


@torch.no_grad()
def predict_test(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    preds = []

    for i, r in enumerate(records):
        if i % 50 == 0:
            print(f"Predicting {i}/{len(records)}")

        candidate_texts = generate_candidates_for_prompt(r["prompt"])
        selected_text, verifier_info = select_best_output(candidate_texts)

        preds.append({
            "id": r["id"],
            "query_vi": r["query_vi"],
            "type": r["type"],
            "model_output": selected_text,
        })

    return preds


# ============================================================
# Case 1: Official test.json exists
# ============================================================
if TEST_PATH.exists():
    print("Found official test.json:", TEST_PATH)

    raw_test = ensure_list_records(read_json_or_jsonl(TEST_PATH))
    test_records = build_test_records(raw_test)
    test_predictions = predict_test(test_records)

    write_json(test_predictions, WORK_DIR / "test_predictions.json")

    print("Saved official prediction file:")
    print(WORK_DIR / "test_predictions.json")
    print(json.dumps(test_predictions[:2], ensure_ascii=False, indent=2))


# ============================================================
# Case 2: No test.json yet -> use valid.json as pseudo-test
# ============================================================
else:
    print("No official test.json found.")
    print("Using valid.json as pseudo-test to check prediction pipeline.")

    raw_pseudo_test = ensure_list_records(read_json_or_jsonl(VALID_PATH))
    pseudo_test_records = build_test_records(raw_pseudo_test)

    pseudo_predictions = predict_test(pseudo_test_records)

    # File có format giống test_predictions.json
    write_json(pseudo_predictions, WORK_DIR / "test_predictions_from_valid.json")

    # Nếu bạn muốn ép tên đúng là test_predictions.json để test submission format
    write_json(pseudo_predictions, WORK_DIR / "test_predictions.json")

    print("Saved pseudo-test prediction files:")
    print(WORK_DIR / "test_predictions_from_valid.json")
    print(WORK_DIR / "test_predictions.json")

    print(json.dumps(pseudo_predictions[:2], ensure_ascii=False, indent=2))

No official test.json found.
Using valid.json as pseudo-test to check prediction pipeline.
Predicting 0/1000
Predicting 50/1000
Predicting 100/1000
Predicting 150/1000
Predicting 200/1000
Predicting 250/1000
Predicting 300/1000
Predicting 350/1000
Predicting 400/1000
Predicting 450/1000
Predicting 500/1000
Predicting 550/1000
Predicting 600/1000
Predicting 650/1000
Predicting 700/1000
Predicting 750/1000
Predicting 800/1000
Predicting 850/1000
Predicting 900/1000
Predicting 950/1000
Saved pseudo-test prediction files:
/kaggle/working/test_predictions_from_valid.json
/kaggle/working/test_predictions.json
[
  {
    "id": 0,
    "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",